# HW18. Снижение размерности: PCA, t-SNE, UMAP

В этой работе я беру `digits`, смотрю PCA для сжатия признаков и сравниваю 2D-проекции PCA, t-SNE и UMAP.

In [ ]:
import inspect
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Данные

In [ ]:
digits = load_digits()
X = digits.data
y = digits.target

print("X shape:", X.shape)
print("Классы:", np.unique(y))

fig, axes = plt.subplots(2, 5, figsize=(8, 3))
for digit, ax in zip(range(10), axes.ravel()):
    idx = np.where(y == digit)[0][0]
    ax.imshow(digits.images[idx], cmap="gray")
    ax.set_title(str(digit))
    ax.axis("off")
plt.tight_layout()
plt.show()

X_scaled = StandardScaler().fit_transform(X)

## 2. PCA: explained variance

In [ ]:
pca_full = PCA(n_components=64, random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)

evr = pca_full.explained_variance_ratio_
cumvar = np.cumsum(evr)

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(cumvar) + 1), cumvar, marker="o", markersize=3)
for level in [0.80, 0.95, 0.99]:
    plt.axhline(level, linestyle="--", label=f"{int(level * 100)}%")
plt.xlabel("Число компонент")
plt.ylabel("Накопленная объяснённая дисперсия")
plt.title("PCA explained variance")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

component_counts = {}
for threshold in [0.80, 0.95, 0.99]:
    component_counts[threshold] = int(np.argmax(cumvar >= threshold) + 1)
    print(f"{int(threshold * 100)}% дисперсии -> {component_counts[threshold]} компонент")

## 3. PCA в 2D

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, s=12, cmap="tab10", alpha=0.8)
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})")
plt.title("Digits в первых двух PCA-компонентах")
plt.colorbar(scatter, ticks=range(10))
plt.tight_layout()
plt.show()

## 4. Loadings первых компонент

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for i, ax in enumerate(axes):
    vmax = np.abs(pca_2d.components_[i]).max()
    img = ax.imshow(pca_2d.components_[i].reshape(8, 8), cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(f"Loadings PC{i + 1}")
    ax.axis("off")
fig.colorbar(img, ax=axes.ravel().tolist(), shrink=0.75)
plt.show()

## 5. PCA как препроцессинг для классификации

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

experiments = {
    "без PCA": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
    "PCA 95%": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
        ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
    "PCA 80%": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.80, random_state=RANDOM_STATE)),
        ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
}

acc_rows = []
for name, pipe in experiments.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    acc_rows.append({"experiment": name, "accuracy": accuracy_score(y_test, pred)})

accuracy_results = pd.DataFrame(acc_rows)
accuracy_results

## 6. t-SNE

Перед t-SNE оставляю 30 PCA-компонент: так обычно быстрее и меньше мелкого шума.

In [ ]:
X_for_tsne = PCA(n_components=30, random_state=RANDOM_STATE).fit_transform(X_scaled)

tsne_params = {
    "n_components": 2,
    "perplexity": 30,
    "learning_rate": "auto",
    "init": "pca",
    "random_state": RANDOM_STATE,
}
if "max_iter" in inspect.signature(TSNE).parameters:
    tsne_params["max_iter"] = 1000
else:
    tsne_params["n_iter"] = 1000

X_tsne_2d = TSNE(**tsne_params).fit_transform(X_for_tsne)

plt.figure(figsize=(7, 5))
plt.scatter(X_tsne_2d[:, 0], X_tsne_2d[:, 1], c=y, s=12, cmap="tab10", alpha=0.8)
plt.title("t-SNE проекция digits")
plt.colorbar(ticks=range(10))
plt.tight_layout()
plt.show()

## 7. UMAP

In [ ]:
try:
    import umap

    X_umap_2d = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        random_state=RANDOM_STATE,
    ).fit_transform(X_scaled)
except Exception as exc:
    print("UMAP недоступен, вместо него использую PCA 2D как fallback:", exc)
    X_umap_2d = X_pca_2d.copy()

plt.figure(figsize=(7, 5))
plt.scatter(X_umap_2d[:, 0], X_umap_2d[:, 1], c=y, s=12, cmap="tab10", alpha=0.8)
plt.title("UMAP проекция digits")
plt.colorbar(ticks=range(10))
plt.tight_layout()
plt.show()

## 8. Сравнение проекций

In [ ]:
projections = {
    "PCA": X_pca_2d,
    "t-SNE": X_tsne_2d,
    "UMAP": X_umap_2d,
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, coords) in zip(axes, projections.items()):
    ax.scatter(coords[:, 0], coords[:, 1], c=y, s=10, cmap="tab10", alpha=0.8)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

silhouette_rows = []
for name, coords in projections.items():
    silhouette_rows.append({"method": name, "silhouette": silhouette_score(coords, y)})
silhouette_results = pd.DataFrame(silhouette_rows).sort_values("silhouette", ascending=False)
silhouette_results

## 9. Реконструкция PCA

In [ ]:
components_to_show = [5, 15, 30, component_counts[0.95]]
sample_idx = np.where(y == 8)[0][0]

fig, axes = plt.subplots(1, len(components_to_show) + 1, figsize=(11, 2.5))
axes[0].imshow(X[sample_idx].reshape(8, 8), cmap="gray")
axes[0].set_title("original")
axes[0].axis("off")

for ax, n_components in zip(axes[1:], components_to_show):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
    ])
    transformed = pipe.fit_transform(X)
    scaler = pipe.named_steps["scaler"]
    pca = pipe.named_steps["pca"]
    restored_scaled = pca.inverse_transform(transformed[sample_idx:sample_idx + 1])
    restored = scaler.inverse_transform(restored_scaled)[0]
    ax.imshow(restored.reshape(8, 8), cmap="gray")
    ax.set_title(f"{n_components} comp.")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Выводы

Для сохранения 95% дисперсии нужно около 40 компонент, то есть размерность можно уменьшить примерно на треть без сильной потери информации. В классификации PCA обычно немного снижает accuracy, зато делает признаки компактнее. PCA полезен как препроцессинг, а t-SNE и UMAP лучше подходят для визуального анализа: они заметно лучше разводят группы цифр на 2D-карте, хотя их координаты уже нельзя интерпретировать так же прямо, как главные компоненты PCA.